# Item-item collaborative filtering trên MovieLens

MovieLens "latest-small": 100 836 rating (0.5–5 sao) của 610 user cho 9 724 phim, 1996–2018. Ta coi
đây là bài toán **implicit feedback**: "user sẽ xem (và chấm) phim nào tiếp theo?", và so hai cách
gợi ý 10 phim: phim phổ biến nhất (popularity) và item-item CF.

Đánh giá offline theo thời gian của từng user: 20% lượt rating **mới nhất** của mỗi user làm test.
Một phim là "trúng" nếu nằm trong tập test của user đó.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`.
- Cell gắn tag `explore` (dòng đầu `# explore`) đọc dữ liệu thật; test bỏ qua.
- Quy ước thứ tự: **điểm bằng nhau thì `movieId` nhỏ hơn đứng trước**. Không gợi ý phim user đã
  rating trong train.
- Chấm: `uv run pytest tracks/data-science/modules/12-advanced-ml/exercises/intermediate/02-movielens-item-item -q`

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp

In [ ]:
# explore: đọc dữ liệu thật (tải bằng: uv run python data/download_dataset.py movielens)
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
ratings = pd.read_csv(ROOT / "data/raw/movielens/ratings.csv")
movies = pd.read_csv(ROOT / "data/raw/movielens/movies.csv")
print(ratings.shape, ratings.userId.nunique(), "user", ratings.movieId.nunique(), "phim")
print("density:", len(ratings) / (ratings.userId.nunique() * ratings.movieId.nunique()))
ratings.groupby("userId").size().describe()

In [ ]:
# explore: long tail. Bao nhiêu phim chỉ có 1 rating? 10% phim phổ biến nhất chiếm bao nhiêu % rating?
counts = ratings.movieId.value_counts()
print("phim chỉ có 1 rating:", (counts == 1).sum())
print("10% phim phổ biến nhất chiếm", round(counts.head(len(counts) // 10).sum() / len(ratings), 3), "số rating")

## 1. Chia train/test theo thời gian của từng user

`split_last_fraction(ratings, frac=0.2)`: với mỗi user, sắp theo `timestamp` (bằng nhau thì theo
`movieId`), lấy $$\lceil \text{frac} \cdot n_u \rceil$$ dòng **cuối** làm test, phần còn lại làm train.
Trả `(train, test)`, giữ nguyên các cột.

In [ ]:
def split_last_fraction(ratings: pd.DataFrame, frac: float = 0.2) -> tuple[pd.DataFrame, pd.DataFrame]:
    """20% (frac) lượt rating mới nhất của mỗi user vào test."""
    # TODO: sort theo userId, timestamp, movieId; đếm thứ tự từ cuối trong mỗi user (cumcount(ascending=False))
    # TODO: test = các dòng có thứ tự từ cuối < ceil(frac * số rating của user)
    raise NotImplementedError

## 2. Popularity baseline

`popularity_recs(train, users, k=10)`: dict `user → list movieId`, phim có nhiều rating nhất trong
train (bằng nhau thì `movieId` nhỏ trước), bỏ phim user đã rating trong train. User không có trong
train vẫn nhận danh sách phổ biến (đây là cách xử lý cold start đơn giản nhất).

In [ ]:
def popularity_recs(train: pd.DataFrame, users, k: int = 10) -> dict:
    """Top-k phim nhiều rating nhất mà user chưa xem."""
    # TODO: thứ tự phim theo (số rating giảm dần, movieId tăng dần)
    # TODO: với mỗi user, bỏ phim đã xem, lấy k phim đầu
    raise NotImplementedError

## 3. Item-item CF

Ma trận nhị phân $$X$$ (user × phim, 1 nếu user đã rating phim trong train). Cosine giữa hai phim là
cosine giữa hai **cột**:

$$
\operatorname{sim}(i, j) = \frac{x_i \cdot x_j}{\lVert x_i \rVert \, \lVert x_j \rVert}
$$

Điểm của phim $$i$$ với user $$u$$ là tổng độ giống của $$i$$ với các phim trong lịch sử $$H_u$$:
$$\text{score}(u, i) = \sum_{j \in H_u} \operatorname{sim}(i, j)$$. Gợi ý k phim điểm cao nhất mà user
chưa xem (bằng điểm thì `movieId` nhỏ trước).

Ma trận độ giống đầy đủ của 8 nghìn phim có 67 triệu ô. Bạn **không cần** tạo nó: nếu $$\tilde X$$ là
$$X$$ đã chuẩn hoá mỗi cột về độ dài 1, thì vector điểm của user là $$\tilde X^\top (\tilde X h_u)$$ với
$$h_u$$ là vector chỉ thị lịch sử. User không có trong train nhận danh sách rỗng.

In [ ]:
def item_item_recs(train: pd.DataFrame, users, k: int = 10) -> dict:
    """Top-k phim theo tổng cosine với lịch sử của user (ma trận nhị phân)."""
    # TODO: mã hoá userId, movieId thành chỉ số; dựng scipy.sparse.csr_matrix nhị phân
    # TODO: chuẩn hoá mỗi cột về độ dài 1
    # TODO: với mỗi user: điểm = Xn.T @ (tổng các cột trong lịch sử), loại phim đã xem, xếp hạng
    raise NotImplementedError

## 4. Đánh giá

`evaluate(recs, test, k=10)` trả `{"precision", "recall", "coverage"}`: precision@k và recall@k
trung bình trên **mọi user có trong `test`** (user thiếu trong `recs` coi như danh sách rỗng), và
coverage là số phim khác nhau xuất hiện trong mọi danh sách gợi ý (cắt ở k).

In [ ]:
def evaluate(recs: dict, test: pd.DataFrame, k: int = 10) -> dict[str, float]:
    """precision@k, recall@k trung bình theo user của test; coverage = số phim khác nhau được gợi ý."""
    # TODO
    raise NotImplementedError

In [ ]:
# explore: so hai cách trên dữ liệu thật
train, test = split_last_fraction(ratings)
users = sorted(test.userId.unique())
pop = popularity_recs(train, users)
ii = item_item_recs(train, users)
pd.DataFrame({"popularity": evaluate(pop, test), "item-item": evaluate(ii, test)}).T

In [ ]:
# explore: nếu chỉ tính "trúng" khi user chấm >= 4 sao thì sao?
liked = test[test.rating >= 4]
pd.DataFrame({"popularity": evaluate(pop, liked), "item-item": evaluate(ii, liked)}).T

In [ ]:
# explore: nhìn một user cụ thể
title = movies.set_index("movieId").title
u = 1
print("Lịch sử gần nhất:", title.reindex(train[train.userId == u].tail(5).movieId).tolist())
print("Popularity:", title.reindex(pop[u][:5]).tolist())
print("Item-item:", title.reindex(ii[u][:5]).tolist())
print("Số phim trúng trong 10 gợi ý: popularity", len(set(pop[u]) & set(test[test.userId == u].movieId)),
      "| item-item", len(set(ii[u]) & set(test[test.userId == u].movieId)))

## 5. Nhận xét (viết vào cell này)

So hai cách theo precision@10, recall@10 và coverage. Kết quả đổi thế nào khi chỉ tính phim được
chấm ≥ 4 sao? Bạn sẽ dùng cách nào cho trang chủ của user mới, và cho user đã xem 200 phim?